<a href="https://colab.research.google.com/github/HST0077/HYOTC/blob/main/PD_estimation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 3년, 5년, 10년 만기에 대한 CDS 스프레드가 각각 50,60,100bp인 기업이 있다. 그리고 이 기업의 회수율은 60%이다. 이 기업의 hazard rate를 연도별로 추정해 보아라.

In [ ]:
import numpy as np
import pandas as pd

# =========================
# 문제 입력
# =========================
R = 0.60  # recovery rate
LGD = 1.0 - R

# CDS par spreads (per annum) at maturities (years)
spreads = {3: 50e-4, 5: 60e-4, 10: 100e-4}

# 누적(0~T) 평균 hazard 계산
avg_h = {T: spreads[T] / LGD for T in spreads}

# 구간 경계 (0, 3, 5, 10)
tenors = [0] + sorted(spreads.keys())

# 구간별 hazard 계산
rows = []
prev_T = 0
prev_int = 0.0
for T in tenors[1:]:
    int_0_T = T * avg_h[T]
    lam_segment = (int_0_T - prev_int) / (T - prev_T)  # 구간 평균 hazard
    rows.append({
        "Segment": f"{prev_T}~{T}y",
        "Hazard rate (p.a.)": lam_segment,
        "Hazard rate (%)": 100 * lam_segment,
    })
    prev_T = T
    prev_int = int_0_T

df = pd.DataFrame(rows)
df

,Segment,Hazard rate (p.a.),Hazard rate (%)
0,0~3y,0.01250,1.250
1,3~5y,0.01875,1.875
2,5~10y,0.03500,3.500


In [ ]:
avg_h

{3: 0.012499999999999999, 5: 0.015, 10: 0.024999999999999998}

In [ ]:
# 구간별 hazard로부터 누적생존확률도 계산해보기
# V(T) = exp(-∫_0^T lambda ds)
cum_int = 0.0
surv_rows = []
for seg in rows:
    a, b = seg["Segment"].replace("y","").split("~")
    a, b = float(a), float(b)
    lam = seg["Hazard rate (p.a.)"]
    cum_int += lam * (b - a)
    surv_rows.append({"T (years)": b, "Survival V(T)": np.exp(-cum_int)})
surv_df = pd.DataFrame(surv_rows)
surv_df

,T (years),Survival V(T)
0,3.0,0.963194
1,5.0,0.927743
2,10.0,0.778801


# (Bond Price) 연 5% 쿠폰을 6개월마다 지급하는 5년 만기 채권 (액면 100)이 있다. 현재 시장이자율이 flat term structure 로 3% 라고 할 때, 이 채권의 현재 가치는 얼마인가?  

In [1]:
import sympy as sp
import numpy as np
import pandas as pd

r = sp.symbols('r', positive=True)
face_value, freq, maturity, coupon_rate = 100, 2, 5, 0.05

# freq: 연 쿠폰 지급횟수
period = 1 / freq

T = np.arange(period, maturity + 0.001, period)
nT = np.arange(1, len(T) + 1)
Disc = 1 / (1 + r/freq) ** nT  # 할인율 적용

# 현금흐름 계산
coupons = coupon_rate * face_value
CF = np.array([coupons / freq] * len(T))
CF[-1] += face_value  # 만기 원금 상환

# 할인 현금흐름
Disc_CF = CF * Disc  # Discounted Cash Flow

# 데이터프레임으로 보기
df = pd.DataFrame({
    'Time': T,
    'CashFlow': CF,
    'DF': Disc,
    'Disc_CF': Disc_CF
})

df

,Time,CashFlow,DF,Disc_CF
0,0.5,2.5,1/(r/2 + 1),2.5/(r/2 + 1)
1,1.0,2.5,(r/2 + 1)**(-2),2.5/(r/2 + 1)**2
2,1.5,2.5,(r/2 + 1)**(-3),2.5/(r/2 + 1)**3
3,2.0,2.5,(r/2 + 1)**(-4),2.5/(r/2 + 1)**4
4,2.5,2.5,(r/2 + 1)**(-5),2.5/(r/2 + 1)**5
5,3.0,2.5,(r/2 + 1)**(-6),2.5/(r/2 + 1)**6
6,3.5,2.5,(r/2 + 1)**(-7),2.5/(r/2 + 1)**7
7,4.0,2.5,(r/2 + 1)**(-8),2.5/(r/2 + 1)**8
8,4.5,2.5,(r/2 + 1)**(-9),2.5/(r/2 + 1)**9
9,5.0,102.5,(r/2 + 1)**(-10),102.5/(r/2 + 1)**10


In [2]:
r_val = 0.03

# 1) DF 열 갱신
df["DF"] = df["DF"].apply(lambda expr: float(sp.N(expr.subs({r: r_val}))))

# 2) Disc_CF 열 갱신
df["Disc_CF"] = df["Disc_CF"].apply(lambda expr: float(sp.N(expr.subs({r: r_val}))))

df

,Time,CashFlow,DF,Disc_CF
0,0.5,2.5,0.985222,2.463054
1,1.0,2.5,0.970662,2.426654
2,1.5,2.5,0.956317,2.390792
3,2.0,2.5,0.942184,2.355461
4,2.5,2.5,0.928260,2.320651
5,3.0,2.5,0.914542,2.286355
6,3.5,2.5,0.901027,2.252567
7,4.0,2.5,0.887711,2.219278
8,4.5,2.5,0.874592,2.186481
9,5.0,102.5,0.861667,88.320891


In [5]:
df.sum()

,0
Time,27.500000
CashFlow,125.000000
DF,9.222185
Disc_CF,109.222185


In [4]:
print('채권의 가격:', df["Disc_CF"].sum())

채권의 가격: 109.22218455185453


In [ ]:
import numpy as np

N=100 # notional
c=0.06 # 채권의 coupon (semiannual)
coupons=np.array([1]*10)*(c/2)
CF=coupons*N # 채권의 쿠폰 현금흐름
CF[-1] += N # 채권의 만기시 원금
CF

array([  3.,   3.,   3.,   3.,   3.,   3.,   3.,   3.,   3., 103.])

In [ ]:
# 무위험 채권의 현가화
discounting=0.05 # 연간 5%
term=np.array(np.arange(0.5,5.1,0.5))
DF=np.exp(-discounting*term)
np.sum(DF*CF) # 채권의 현재 가격

np.float64(104.09356799388404)

In [ ]:
# 회사채권의 현가화
discounting=0.07 # 연간 7%
DF=np.exp(-discounting*term)
np.sum(DF*CF) # 회사채권의 현재 가격

np.float64(95.34087448559137)

In [ ]:
# 채권 가격 구하는 함수

import math

def bond_price_continuous_discount(
    face: float,
    coupon_rate_annual: float,
    T: float,
    r_cont_annual: float,
    freq: int = 2
) -> float:
    """
    연속복리 할인(continuous compounding)으로 채권가격 계산.

    Parameters
    ----------
    face : float
        액면가(Face value), 예: 100
    coupon_rate_annual : float
        연 쿠폰율(예: 0.05는 연 5%), semi-annual coupon이면 여전히 '연 쿠폰율'로 입력
    T : float
        만기(년), 예: 5.0
    r_cont_annual : float
        연속복리 무위험이자율(예: 0.04는 4%)
    freq : int
        연 쿠폰 지급 횟수(반기=2)

    Returns
    -------
    float
        채권 현재가격
    """
    # 반기 쿠폰액 (연 쿠폰율을 freq로 나눔)
    c = face * coupon_rate_annual / freq

    # 지급시점들: 0.5, 1.0, ... , T  (T가 freq의 배수라고 가정)
    n_pay = int(round(T * freq))
    times = [(i + 1) / freq for i in range(n_pay)]

    # 각 현금흐름(마지막에는 원금 포함)
    pv = 0.0
    for t in times:
        cash = c
        if abs(t - T) < 1e-12:
            cash += face
        pv += cash * math.exp(-r_cont_annual * t)  # 연속복리 할인

    return pv

In [ ]:
bond_price_continuous_discount(100,0.06,5,0.05,2)

104.09356799388404